# RAG知识库功能使用示例

本Notebook展示如何在现有的word_gen_system中使用RAG知识库功能。

## 1. 环境检查与依赖安装

首先检查必要的依赖是否已安装：

In [ ]:
# 检查依赖
import sys
print('Python版本:', sys.version[:5])

# 尝试导入必要库
try:
    import sentence_transformers
    print('✓ sentence-transformers 已安装')
except ImportError:
    print('✗ sentence-transformers 未安装')
    print('请运行: pip install sentence-transformers')

try:
    import numpy
    print('✓ numpy 已安装')
except ImportError:
    print('✗ numpy 未安装')
    print('请运行: pip install numpy')

try:
    import sklearn
    print('✓ scikit-learn 已安装')
except ImportError:
    print('✗ scikit-learn 未安装')
    print('请运行: pip install scikit-learn')

## 2. 设置RAG功能

使用猴子补丁方法为现有系统添加RAG功能：

In [ ]:
# 设置RAG功能
from rag_monkey_patch import setup_rag_for_notebook

rag_success = setup_rag_for_notebook()

if rag_success:
    print('\n✓ RAG功能已成功设置！')
else:
    print('\n✗ RAG功能设置失败，请检查错误信息')

## 3. 基础RAG功能测试

测试知识库检索功能：

In [ ]:
# 测试知识库检索
from rag_integration import create_rag_manager

rag_manager = create_rag_manager()
if rag_manager:
    print('可用知识库:', rag_manager.list_available_knowledge_bases())
    
    # 测试检索
    query = '不良贷款认定标准'
    results = rag_manager.search_knowledge_base('puhui_daikou_buliang', query, top_k=2)
    
    print(f'\n查询: {query}')
    if results:
        for i, res in enumerate(results, 1):
            print(f'  [{i}] 相似度: {res["similarity"]:.3f}')
            print(f'      文档: {res["doc_name"]}')
            print(f'      内容: {res["content"][:100]}...')
    else:
        print('  未找到相关结果')
else:
    print('RAG管理器创建失败')

## 4. Prompt增强测试

测试{kb:}标记的处理：

In [ ]:
# 测试Prompt增强
test_context = {
    'data': {
        'bad_mom': 0.12,
        'bad_yoy': 0.35,
        'branch_name': 'A支行'
    },
    'loan_summary': {
        'loan_balance': 1250.0
    }
}

test_prompts = [
    # 无知识库引用
    '基于当前不良率{{data.bad_mom}}，请分析风险情况。',
    
    # 简单知识库引用
    '基于{kb:puhui_daikou_buliang}知识库，分析当前风险情况。',
    
    # 带查询词的知识库引用
    '请结合{kb:puhui_daikou_buliang:不良贷款处置方式}，给出针对{{data.branch_name}}的建议。',
    
    # 多个知识库引用
    '基于{kb:puhui_daikou_buliang}和{kb:puhui_daikou_buliang:风险缓释措施}，制定全面方案。'
]

from jinja2 import Environment, StrictUndefined
env = Environment(undefined=StrictUndefined)

for i, prompt_tpl in enumerate(test_prompts, 1):
    print(f'测试 {i}: {prompt_tpl[:50]}...')
    
    # 基本渲染
    basic = env.from_string(prompt_tpl).render(test_context)
    print(f'  基本渲染: {basic[:80]}...')
    
    # RAG增强
    if rag_manager:
        enhanced, used_rag = rag_manager.render_prompt_with_rag(prompt_tpl, test_context)
        print(f'  RAG增强: {used_rag}')
        if used_rag:
            print(f'  增强后长度: {len(enhanced)} 字符')
            print(f'  预览: {enhanced[:150]}...')
    print()

## 5. Word模板示例

创建带有RAG引用的Word模板：

In [ ]:
# 创建带有RAG引用的示例模板
from docx import Document
from pathlib import Path

rag_template_path = Path('templates') / 'template_with_rag.docx'
rag_template_path.parent.mkdir(exist_ok=True)

doc = Document()
doc.add_heading('普惠支行报告（RAG增强版）', level=1)

# 添加普通变量
doc.add_paragraph('支行：{{ data.branch_name }}，报告期：{{ data.report_date }}')
doc.add_paragraph('普惠贷款余额为 {{ loan_summary.loan_balance }} 万元')

# 添加RAG增强的AI段落
doc.add_paragraph('风险分析（以下段落将引用知识库）：')
ai_para = doc.add_paragraph('§AIBLOCK0§基于不良率{{ data.bad_mom }}，结合知识库进行分析。')

# 添加批注（包含{kb:}标记）
comment_text = (
    'prompt="基于不良率{{ data.bad_mom }}数据，结合{kb:puhui_daikou_buliang}知识库，'
    '分析该支行风险状况，给出针对性建议"'
)
try:
    doc.add_comment(runs=ai_para.runs, text=comment_text, author='AI+RAG', initials='AR')
except:
    print('注意：当前python-docx版本可能不支持add_comment')

# 保存模板
doc.save(rag_template_path)
print(f'已创建RAG增强模板: {rag_template_path}')
print(f'批注内容预览: {comment_text[:100]}...')

## 6. 完整流程测试

测试从数据到生成的完整流程：

In [ ]:
# 模拟完整的文档生成流程
print('模拟文档生成流程（使用RAG）...')
print('=' * 60)

# 模拟上下文数据
sample_context = {
    'data': {
        'branch_name': 'A支行',
        'report_date': '2024-12-19',
        'bad_mom': 0.12,
        'bad_yoy': 0.35
    },
    'loan_summary': {
        'loan_balance': 1250.0,
        'house_eval_amount': 900.0
    },
    'products': [
        {'product_name': '法人e抵', 'overdue_rate': 1.2},
        {'product_name': '个人e抵押', 'overdue_rate': 0.4}
    ]
}

# 示例prompt（带RAG引用）
rag_prompt_template = """
基于{{data.branch_name}}的不良率数据（环比{{data.bad_mom}}，同比{{data.bad_yoy}}），
结合{kb:puhui_daikou_buliang:不良贷款处置方式}知识库内容，
请分析该支行的风险状况，并给出具体的处置建议。

要求：
1. 引用相关知识库内容
2. 结合具体数据进行分析
3. 给出可操作的改进建议
"""

print(f'原始Prompt模板（带RAG引用）:')
print('-' * 40)
print(rag_prompt_template)
print('-' * 40)

# 使用RAG增强
if rag_manager:
    enhanced_prompt, used_rag = rag_manager.render_prompt_with_rag(rag_prompt_template, sample_context)
    
    print(f'\nRAG增强结果 (used_rag={used_rag}):')
    print('-' * 40)
    print(enhanced_prompt[:500] + '...' if len(enhanced_prompt) > 500 else enhanced_prompt)
    print('-' * 40)
    
    # 分析增强效果
    print(f'\n效果分析:')
    print(f'  原始长度: {len(rag_prompt_template)} 字符')
    print(f'  增强长度: {len(enhanced_prompt)} 字符')
    print(f'  增加内容: {len(enhanced_prompt) - len(rag_prompt_template)} 字符')
    
    # 检查是否还有{kb:}标记
    import re
    remaining_kb_refs = re.findall(r'\{kb:[^}]+\}', enhanced_prompt)
    if remaining_kb_refs:
        print(f'  警告: 仍有未处理的RAG引用: {remaining_kb_refs}')
    else:
        print(f'  ✓ 所有RAG引用已成功处理')
else:
    print('RAG管理器不可用，无法测试增强功能')

## 7. 使用指南

### 7.1 在现有系统中使用RAG

1. **安装依赖**：
   ```bash
   pip install sentence-transformers numpy scikit-learn python-docx PyPDF2
   ```

2. **创建知识库**：
   ```python
   python create_puhui_knowledge_base.py
   ```

3. **在notebook中启用RAG**：
   ```python
   from rag_monkey_patch import setup_rag_for_notebook
   setup_rag_for_notebook()
   ```

### 7.2 在Word模板中使用

在Word模板的批注中添加{kb:}标记：
```
prompt="基于不良率{{data.bad_mom}}，结合{kb:puhui_daikou_buliang}知识库，分析风险状况"
```

支持两种格式：
- `{kb:知识库名称}`：使用整个prompt作为查询词
- `{kb:知识库名称:具体查询词}`：使用指定的查询词

### 7.3 高级用法

```python
# 手动使用RAG管理器
from rag_integration import create_rag_manager
rag_manager = create_rag_manager()

# 搜索知识库
results = rag_manager.search_knowledge_base('puhui_daikou_buliang', '不良贷款', top_k=3)

# 手动增强prompt
enhanced_prompt, used_rag = rag_manager.enhance_prompt_with_rag(original_prompt, context)
```

### 7.4 故障排除

1. **网络问题**：sentence-transformers需要下载模型，如果网络不通，可以：
   - 使用离线模型
   - 配置代理
   - 使用关键词匹配模式（在rag_module中实现）

2. **知识库加载失败**：
   - 检查knowledge_bases目录结构
   - 确保metadata.json文件存在且格式正确
   - 确保Word文档可以正常读取

3. **RAG增强无效**：
   - 检查{kb:}标记格式是否正确
   - 检查知识库名称是否匹配
   - 查看控制台输出，了解处理过程

## 8. 扩展知识库

要添加新的知识库，只需在`knowledge_bases/`目录下创建新文件夹：

```
knowledge_bases/
├── puhui_daikou_buliang/          # 普惠贷后不良
│   ├── metadata.json
│   ├── 文档1.docx
│   └── 文档2.docx
├── risk_management/               # 风险管理（新知识库）
│   ├── metadata.json
│   └── 风险管理指南.docx
└── regulatory_policies/           # 监管政策（新知识库）
    ├── metadata.json
    └── 最新监管政策.docx
```

然后在Word模板中使用：
```
prompt="结合{kb:risk_management}知识库，制定风险控制方案"
```